# Ozone and temperature

We use EPA ozone data and NOAA weather data to look for periods of elevated
ozone and describe the temperature and data coverage around them.
Alabama is our first example. Georgia will show whether we can use the same
pipeline with a different site and weather station.

So far, we have downloaded the inputs (Bronze) and checked their availability.
Silver will clean and combine the observations using Spark. Gold will produce
the ozone period catalogue and coverage summaries.


## 1. Setup

We start by loading the settings and finding the project folder. Source URLs
and descriptions are in `config/sources.json`.

Local runs use `environments.local.json`. On the VM, set `PIPELINE_TARGET=vm`
in the kernel before running this cell to use `environments.json`.
`PIPELINE_ENVIRONMENT` defaults to `dev`, and `PIPELINE_REFRESH_SOURCES`
defaults to `false`, which lets us reuse files we already have.

If a source has more than one saved version, choose its checksum in
`source_versions`. Otherwise, we use the single available version.


In [1]:
import calendar
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import urlsplit
from zipfile import ZipFile

# Find the project whether Jupyter starts here or in the notebooks folder.
for folder in (Path.cwd(), *Path.cwd().parents):
    project_dir = folder if folder.name == "final_project" else folder / "final_project"
    if (project_dir / "config" / "sources.json").is_file():
        project_dir = project_dir.resolve()
        break
else:
    raise FileNotFoundError("Start Jupyter from inside the project checkout.")

sys.path.insert(0, str(project_dir / "src"))
from run_context import create_run_context

target = os.environ.get("PIPELINE_TARGET", "local")
environment = os.environ.get("PIPELINE_ENVIRONMENT", "dev")
refresh_setting = os.environ.get("PIPELINE_REFRESH_SOURCES", "false").lower()
if target not in {"local", "vm"} or refresh_setting not in {"true", "false"}:
    raise ValueError("Use target local/vm and refresh true/false.")
refresh_sources = refresh_setting == "true"

config_name = "environments.local.json" if target == "local" else "environments.json"
environments_file = project_dir / "config" / config_name
sources = json.loads((project_dir / "config" / "sources.json").read_text(encoding="utf-8"))

# Fill this in only if a source has more than one saved version.
source_versions = {}
# Example: source_versions["epa/hourly_ozone_2025"] = "<original SHA-256>"


### Columns we need

These lists check that each input has the columns needed later in the project.
Other columns are kept too. This checks the file structure, not the quality
of each observation.


In [2]:
# Extra columns are fine; these are the ones we need later.
required_columns = {
    ("epa", "sites"): [
        "State Code", "County Code", "Site Number",
        "Latitude", "Longitude", "Datum",
    ],
    ("epa", "monitors"): [
        "State Code", "County Code", "Site Number",
        "Parameter Code", "POC",
        "First Year of Data", "Last Sample Date",
    ],
    ("epa", "hourly_ozone"): [
        "State Code", "County Code", "Site Num", "Parameter Code", "POC",
        "Date GMT", "Time GMT", "Sample Measurement", "Units of Measure",
        "Qualifier", "MDL", "Method Type", "Method Code",
    ],
    ("noaa", "stations"): [
        "GHCN_ID", "LATITUDE", "LONGITUDE", "ELEVATION", "ISO_CODE",
    ],
    ("noaa", "hourly_weather"): [
        "STATION", "DATE", "temperature",
        "temperature_Measurement_Code", "temperature_Quality_Code",
        "temperature_Report_Type", "temperature_Source_Code",
        "temperature_Source_Station_ID",
    ],
}

## 2. Bronze: download and keep the inputs

We keep the original files so we can trace later results back to their inputs.
EPA ZIPs stay together with their extracted CSVs. A manifest records the
source URL, download details, file sizes, checksums and column check.

### 2.1 File details and column checks

These helpers calculate checksums and put the file details into dictionaries.
We read large files in chunks rather than loading them all into memory.


In [3]:
def file_sha256(path):
    # The ozone CSV is large, so read it a chunk at a time.
    checksum = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            checksum.update(chunk)
    return checksum.hexdigest()


def write_json(path, value):
    """Save to a temporary file first, then replace the JSON file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def source_name(source):
    name = source["dataset"]
    for field in ("year", "station_id"):
        if field in source:
            name += "_" + str(source[field])
    return name


def check_columns(source, columns):
    required = required_columns[(source["source"], source["dataset"])]
    missing = sorted(set(required) - set(columns))
    return {
        "contract_version": "bronze-columns-v2",
        "observed_columns": columns,
        "status": "blocked" if missing else "compatible",
        "findings": [f"Missing required columns: {missing}"] if missing else [],
    }


def file_record(path, checksum, root):
    """File details for the source manifest."""
    return {
        "path": path.relative_to(root).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": checksum,
    }


We save the settings used for each run. In a Git checkout, we also record
the commit and whether there are uncommitted changes. A copied VM folder
is labelled `manual_copy`, with the Git fields left unknown. This tells us
what we can trace back later.


In [4]:
def capture_provenance(context):
    config_path = context.reports_dir / "effective_config.json"
    write_json(config_path, {
        "sources": sources, "target": target, "environment": context.environment,
        "environment_root": str(context.environment_root),
        "refresh_sources": refresh_sources, "source_versions": source_versions,
    })

    # A copied VM folder has no Git history to read, so these values stay unknown.
    provenance = {
        "code_source": "manual_copy",
        "code_revision": None,
        "working_tree_dirty": None,
        "config_sha256": file_sha256(config_path),
        "evidence_path": context.reports_dir.relative_to(context.environment_root).as_posix(),
    }
    in_git_checkout = any(
        (folder / ".git").exists() for folder in (project_dir, *project_dir.parents)
    )
    if in_git_checkout:
        changes = subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=project_dir, text=True
        ).strip()
        provenance.update({
            "code_source": "git",
            "code_revision": subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=project_dir, text=True
            ).strip(),
            "working_tree_dirty": bool(changes),
        })
    return provenance


### 2.2 Download and extract files

Downloads get up to three attempts. We check that the file is not empty and,
when the server gives a size, that the number of downloaded bytes matches.
EPA ZIPs are then extracted; the NOAA CSV and PSV files are used directly.

On a normal rerun, we check the saved file sizes and reuse their recorded
column checks. We do not calculate all checksums again. The saved inputs
should be left unchanged: this size check would miss an edit of the same size.


In [5]:
def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Rename the partial file only after the download checks pass.
    partial = destination.with_name(destination.name + ".part")
    for attempt in range(1, 4):
        try:
            request = Request(url, headers={"Accept-Encoding": "identity"})
            with urlopen(request, timeout=60) as response:
                expected_size = response.headers.get("Content-Length")
                resolved_url = response.geturl()
                with partial.open("wb") as handle:
                    shutil.copyfileobj(response, handle, length=1024 * 1024)
            size = partial.stat().st_size
            if size == 0 or (expected_size is not None and size != int(expected_size)):
                raise ValueError("Download is empty or has an unexpected size.")
            partial.rename(destination)
            return {
                "requested_url": url, "resolved_url": resolved_url,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            }
        except (OSError, ValueError) as error:
            if attempt == 3:
                raise
            print(f"Retry {attempt}: {error}")
            time.sleep(2)


def read_header(path):
    delimiter = "|" if path.suffix == ".psv" else ","
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return next(csv.reader(handle, delimiter=delimiter), [])


def prepare_file(path):
    """Extract the CSV from an EPA ZIP, or return the plain NOAA file."""
    if path.suffix != ".zip":
        return path
    csv_path = path.with_suffix(".csv")
    with ZipFile(path) as archive:
        with archive.open(csv_path.name) as source, csv_path.open("xb") as destination:
            shutil.copyfileobj(source, destination)
    return csv_path


def stored_data_path(manifest, context, verify_hashes=False):
    """Find the saved data file and check that its files are still present."""
    for record in [manifest["original_file"], *manifest["extracted_files"]]:
        path = context.environment_root / record["path"]
        if not path.is_file() or path.stat().st_size != record["size_bytes"]:
            raise ValueError(f"Stored file is missing or its size changed: {path}")
        if verify_hashes and file_sha256(path) != record["sha256"]:
            raise ValueError(f"Stored file checksum differs: {path}")
    if manifest["extracted_files"]:
        data_file = manifest["extracted_files"][0]
    else:
        data_file = manifest["original_file"]
    return context.environment_root / data_file["path"]

### 2.3 Download or reuse a source

We first look for a saved version. If there is none, or refresh is enabled,
we download the file, check its columns and save it under its checksum.

If the download matches a version we already have, we keep the original
manifest. Once the saved copy is verified, we remove this run's extra copy.
Complete files with missing required columns are kept but marked as blocked
for later processing.


In [6]:
def acquire_source(source, context, provenance):
    """Find a saved source version or download it, then return its details."""
    name = source_name(source)
    key = source["source"] + "/" + name
    published = context.published_root / "bronze" / source["source"] / name
    version = source_versions.get(key)
    manifest_path = None
    identity = {"provider": source["source"], "dataset": source["dataset"]}
    for field in ("year", "station_id"):
        if field in source:
            identity[field] = source[field]

    if not refresh_sources:
        if version is not None:
            manifest_path = published / version / "manifest.json"
            if not manifest_path.is_file():
                raise FileNotFoundError(f"Selected source version is missing: {manifest_path}")
        else:
            candidates = sorted(published.glob("*/manifest.json"))
            if len(candidates) > 1:
                raise ValueError(f"Select a checksum in source_versions for {key}.")
            manifest_path = candidates[0] if candidates else None

    downloaded = manifest_path is None
    acquisition = None
    if not downloaded:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        data_path = stored_data_path(manifest, context)
        outcome = "reused"
    else:
        staging = context.staging_root / "bronze" / source["source"] / name
        filename = Path(urlsplit(source["url"]).path).name
        original = staging / filename
        acquisition = download_file(source["url"], original)
        acquisition["run_id"] = context.run_id
        data = prepare_file(original)
        schema = check_columns(source, read_header(data))
        checksum = file_sha256(original)
        data_checksum = checksum if data == original else file_sha256(data)
        snapshot = context.data_root / "bronze" / source["source"] / name / checksum
        reused = snapshot.exists()
        if not reused:
            snapshot.parent.mkdir(parents=True, exist_ok=True)
            staging.rename(snapshot)

        # The manifest points to the saved files, not the staging folder.
        original_record = file_record(snapshot / original.name, checksum, context.environment_root)
        extracted = []
        if data != original:
            extracted_record = file_record(snapshot / data.name, data_checksum, context.environment_root)
            extracted_record["archive_member"] = data.name
            extracted.append(extracted_record)
        candidate = {
            "manifest_version": 1, "source": identity, "acquisition": acquisition,
            "original_file": original_record, "extracted_files": extracted,
            "schema": schema, "provenance": provenance,
        }
        if reused:
            stored_data_path(candidate, context, verify_hashes=True)
        manifest_path = published / checksum / "manifest.json"
        if manifest_path.exists():
            # Keep the original download details when the version already exists.
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        else:
            manifest = candidate
            write_json(manifest_path, manifest)
        data_path = snapshot / data.name
        outcome = "reused" if reused else "acquired"

        # Remove only this run's extra copies after the saved version is checked.
        if reused:
            if not staging.resolve().is_relative_to(context.staging_root.resolve()):
                raise ValueError("Duplicate cleanup must stay inside this run's staging.")
            for path in {original, data}:
                path.unlink()
            if not any(staging.iterdir()):
                staging.rmdir()

    return {
        "request": source, "manifest_path": manifest_path, "data_path": data_path,
        "schema": manifest["schema"], "outcome": outcome, "downloaded": downloaded,
        "acquisition": acquisition,
    }

### 2.4 Run acquisition

We process the sources one at a time and save a short run report. The lock
folder stops two acquisition runs from writing to the same environment at
once. If a source fails, the report records the failure and the notebook stops.


In [7]:
def run_bronze():
    context = create_run_context(
        environments_file, environment,
        test_run_id=os.environ.get("PIPELINE_TEST_RUN_ID"),
    )
    context.reports_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    report = {
        "run_id": context.run_id, "environment": context.environment,
        "status": "failure", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "refresh_sources": refresh_sources, "results": [], "blocked_sources": [],
    }
    downloads = []
    lock = context.environment_root / ".bronze-publication.lock"
    locked = False
    print(f"Target: {target}; environment: {environment}; refresh: {refresh_sources}")
    print(f"Run: {context.run_id}")
    try:
        provenance = capture_provenance(context)
        report["provenance"] = provenance
        # Allow only one acquisition run to write to this environment at a time.
        lock.mkdir()
        locked = True
        for source in sources:
            try:
                item = acquire_source(source, context, provenance)
            except Exception as error:
                report["results"].append({
                    "request": source, "outcome": "failed", "error": str(error),
                })
                raise
            downloads.append(item)
            report["results"].append({
                "request": source, "outcome": item["outcome"],
                "downloaded": item["downloaded"],
                "manifest_path": item["manifest_path"].relative_to(
                    context.environment_root
                ).as_posix(),
                "acquisition": item["acquisition"], "error": None,
            })
            if item["schema"]["status"] == "blocked":
                report["blocked_sources"].append(source)
            action = "Downloaded" if item["downloaded"] else "Reused"
            print(f"{action}: {source['source']}/{source_name(source)} "
                  f"| {item['schema']['status']} | {item['data_path']}")
        report["status"] = "success"
    except Exception as error:
        report["error"] = str(error)
        raise
    finally:
        # Save a report even if one of the sources failed.
        if locked:
            lock.rmdir()
        report["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        report["elapsed_seconds"] = round(time.perf_counter() - started, 3)
        report_path = context.reports_dir / "run.json"
        write_json(report_path, report)
        print(f"Run report: {report_path}")
    return downloads

The next cell starts acquisition. With refresh set to `false`, it reuses
saved sources and downloads any that are missing. All six inputs are already
available in our current WSL and VM development folders.

For a full check, save the notebook, restart the kernel and run from the top
with the right local or VM settings.


In [8]:
downloads = run_bronze()

Target: local; environment: dev; refresh: False
Run: d7362c705b90421998ce43c9125ffb2a
Reused: epa/sites | compatible | /home/amund/dat535-final-project-data/dev/data/bronze/epa/sites/56b322f6a3d5368d055bce53d0988275c0b27fbcced3d57a6ce8f2e2b539e15a/aqs_sites.csv
Reused: epa/monitors | compatible | /home/amund/dat535-final-project-data/dev/data/bronze/epa/monitors/e8325216d4ec5df462a23c201e6ffa1176b2fce363c9f6b0e3f18ce6ad958525/aqs_monitors.csv
Reused: epa/hourly_ozone_2025 | compatible | /home/amund/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.csv
Reused: noaa/stations | compatible | /home/amund/dat535-final-project-data/dev/data/bronze/noaa/stations/812ea50bd5efd1b9cc09d5800c55825b3d43b3c5aa75ab17b2848f43b1bfaaef/ghcnh-station-list.csv
Reused: noaa/hourly_weather_2025_USW00053864 | compatible | /home/amund/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864/2

## Shared availability check

We use one function for Alabama and Georgia. It reads the files selected by
a profile, counts available hours and shows the temperature codes.
Each call scans the annual ozone CSV once, so it may take a little time.

The table shows numeric ozone hours, numeric temperature hours and temperature
hours passing our trial source/quality rules (`223/1` and `343/5`). Sets count
each hour once even when it has several weather reports. Code counts, on the
other hand, count reports, including those with missing temperature.

We compare timestamps in UTC. A weather report at 14:56 counts towards the
14:00 hour for this availability check. Overlap is shared hours divided by
observed ozone hours; `n/a` means there were no ozone hours that month.
Ozone qualifiers are not filtered here.

[NOAA's documentation](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf)
explains the UTC timestamps, Celsius temperatures and source-dependent codes.
We will do the cleaning and hourly temperature selection with Spark in
Silver.


In [9]:
def check_region(profile, downloads):
    """Count available hours for one monitor and weather station."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_id = "-".join(monitor[field] for field in (
        "state_code", "county_code", "site_number", "parameter_code", "poc"
    ))
    station = profile["weather_mapping"]["noaa_station_id"]
    trial_codes = {
        tuple(pair) for pair in profile["temperature_policy"]["source_quality_pairs"]
    }

    # Find the two inputs selected by this profile.
    ozone = next(
        item for item in downloads
        if item["request"]["source"] == "epa"
        and item["request"]["dataset"] == "hourly_ozone"
        and item["request"].get("year") == year
    )
    weather = next(
        item for item in downloads
        if item["request"]["source"] == "noaa"
        and item["request"]["dataset"] == "hourly_weather"
        and item["request"].get("year") == year
        and item["request"].get("station_id") == station
    )
    for item in (ozone, weather):
        if item["schema"]["status"] != "compatible":
            raise ValueError(f"Source is blocked: {item['data_path']}")

    # Sets stop several reports in the same hour being counted more than once.
    ozone_hours = {month: set() for month in range(1, 13)}
    numeric_temp_hours = {month: set() for month in range(1, 13)}
    trial_temp_hours = {month: set() for month in range(1, 13)}
    temperature_codes = Counter()
    monitor_fields = ["State Code", "County Code", "Site Num", "Parameter Code", "POC"]

    with ozone["data_path"].open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            if "-".join(row[field].strip() for field in monitor_fields) != monitor_id:
                continue
            stamp = datetime.fromisoformat(row["Date GMT"] + "T" + row["Time GMT"])
            if stamp.year != year:
                continue
            try:
                value = float(row["Sample Measurement"])
            except ValueError:
                continue
            if math.isfinite(value):
                ozone_hours[stamp.month].add(stamp)

    with weather["data_path"].open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle, delimiter="|"):
            if row["STATION"].strip() != station:
                continue
            stamp = datetime.fromisoformat(row["DATE"])
            if stamp.year != year:
                continue
            source = row["temperature_Source_Code"].strip()
            quality = row["temperature_Quality_Code"].strip()
            report_type = row["temperature_Report_Type"].strip()
            # Show these codes even when the temperature value is missing.
            temperature_codes[(source, quality, report_type)] += 1
            try:
                value = float(row["temperature"])
            except ValueError:
                continue
            if not math.isfinite(value):
                continue
            # Group by clock hour for availability; no temperature value is chosen here.
            hour = stamp.replace(minute=0, second=0, microsecond=0)
            numeric_temp_hours[stamp.month].add(hour)
            if (source, quality) in trial_codes:
                trial_temp_hours[stamp.month].add(hour)

    # Include months without observations, so gaps stay visible.
    monthly = []
    for month in range(1, 13):
        ozone_count = len(ozone_hours[month])
        shared = len(ozone_hours[month] & trial_temp_hours[month])
        monthly.append({
            "month": month,
            "calendar_hours": calendar.monthrange(year, month)[1] * 24,
            "ozone_hours": ozone_count,
            "numeric_temperature_hours": len(numeric_temp_hours[month]),
            "trial_temperature_hours": len(trial_temp_hours[month]),
            "shared_hours": shared,
            "ozone_without_temperature": ozone_count - shared,
            "overlap_percent": 100 * shared / ozone_count if ozone_count else None,
        })

    print(f"{profile['region']} {year} ({profile['status']})")
    print(f"Ozone monitor: {monitor_id}; weather station: {station}")
    print("\nTemperature source | Quality | Report type | Reports")
    for codes, count in temperature_codes.most_common():
        print(" | ".join(code or "<blank>" for code in codes), "|", count)

    print("\nMonth | Calendar | Ozone | Numeric temp | Trial temp | Shared | Without temp | Overlap")
    for row in monthly:
        percent = row["overlap_percent"]
        percent_text = f"{percent:.1f}%" if percent is not None else "n/a"
        print(f"{row['month']:02d} | {row['calendar_hours']} | {row['ozone_hours']} | "
              f"{row['numeric_temperature_hours']} | {row['trial_temperature_hours']} | "
              f"{row['shared_hours']} | {row['ozone_without_temperature']} | {percent_text}")
    total_ozone = sum(row["ozone_hours"] for row in monthly)
    total_shared = sum(row["shared_hours"] for row in monthly)
    total_missing = sum(row["ozone_without_temperature"] for row in monthly)
    print(f"\nTotals: {total_ozone} ozone hours; {total_shared} shared hours; "
          f"{total_missing} without temperature")

    return {
        "monthly": monthly,
        "temperature_codes": dict(temperature_codes),
        "source_manifests": {
            "ozone": ozone["manifest_path"], "weather": weather["manifest_path"],
        },
    }


## 3. Alabama

We use McAdory ozone and Alabaster temperature for 2025. The weather station
is about 26.7 km away, so its observations give us nearby weather context.

### 3.1 Load the profile

The JSON file holds the monitor, weather station and trial temperature codes.


In [10]:
alabama_profile = json.loads(
    (project_dir / "config" / "alabama_profile.json").read_text(encoding="utf-8")
)


### 3.2 Check availability

We pass the Alabama profile and the downloaded inputs to the shared function.
The result is also kept in `alabama_check`.


In [11]:
alabama_check = check_region(alabama_profile, downloads)


Alabama 2025 (provisional)
Ozone monitor: 01-073-1005-44201-1; weather station: USW00053864

Temperature source | Quality | Report type | Reports
343 | 5 | FM15 | 5556
223 | 1 | FM15 | 2957
343 | 5 | FM16 | 2327
223 | 1 | FM16 | 846
999 | <blank> | FM15 | 121
<blank> | <blank> | <blank> | 29
999 | <blank> | FM16 | 8
343 | 6 | FM16 | 6
223 | 2 | FM15 | 1

Month | Calendar | Ozone | Numeric temp | Trial temp | Shared | Without temp | Overlap
01 | 744 | 0 | 738 | 738 | 0 | 0 | n/a
02 | 672 | 89 | 672 | 672 | 89 | 0 | 100.0%
03 | 744 | 6 | 744 | 744 | 6 | 0 | 100.0%
04 | 720 | 684 | 720 | 720 | 684 | 0 | 100.0%
05 | 744 | 709 | 744 | 744 | 709 | 0 | 100.0%
06 | 720 | 705 | 697 | 697 | 683 | 22 | 96.9%
07 | 744 | 725 | 744 | 744 | 725 | 0 | 100.0%
08 | 744 | 737 | 739 | 667 | 660 | 77 | 89.6%
09 | 720 | 715 | 702 | 655 | 651 | 64 | 91.0%
10 | 744 | 735 | 671 | 671 | 663 | 72 | 90.2%
11 | 720 | 61 | 719 | 719 | 61 | 0 | 100.0%
12 | 744 | 0 | 744 | 743 | 0 | 0 | n/a

Totals: 5166 ozone hours;

### 3.3 Findings

There are 5,166 ozone hours and temperature in 4,931 of those hours (95.5%).
The 235 hours without trial temperature are in June, August, September and
October. January and December have no ozone observations.

February and March have very little ozone data. Those 95 observations have
qualifier `6`, which we agreed to exclude in Silver. They are still included
in this availability check.


## 4. Georgia

We try South DeKalb ozone with Atlanta Hartsfield–Jackson temperature for
2025. The weather station is about 15.5 km from the ozone site.

This is our CP2 availability check. We will move the full Georgia rerun
demonstration after Alabama's Silver and Gold sections when those are ready.

### 4.1 Load the profile

The file has the same structure as Alabama's, with different site and station
IDs. It uses the same trial temperature codes.


In [12]:
georgia_profile = json.loads(
    (project_dir / "config" / "georgia_profile.json").read_text(encoding="utf-8")
)


### 4.2 Check availability

We call the same function with the Georgia profile and keep its result in
`georgia_check`.


In [13]:
georgia_check = check_region(georgia_profile, downloads)


Georgia 2025 (provisional)
Ozone monitor: 13-089-0002-44201-1; weather station: USW00013874

Temperature source | Quality | Report type | Reports
343 | 5 | FM15 | 5553
223 | 1 | FM15 | 3057
223 | 1 | FM12 | 2857
343 | 5 | FM16 | 1119
223 | 1 | FM16 | 424
999 | <blank> | FM15 | 121
343 | 7 | FM15 | 15
<blank> | <blank> | <blank> | 2
999 | <blank> | FM16 | 1

Month | Calendar | Ozone | Numeric temp | Trial temp | Shared | Without temp | Overlap
01 | 744 | 724 | 742 | 742 | 722 | 2 | 99.7%
02 | 672 | 661 | 672 | 672 | 661 | 0 | 100.0%
03 | 744 | 717 | 744 | 744 | 717 | 0 | 100.0%
04 | 720 | 706 | 720 | 720 | 706 | 0 | 100.0%
05 | 744 | 734 | 744 | 744 | 734 | 0 | 100.0%
06 | 720 | 702 | 720 | 719 | 701 | 1 | 99.9%
07 | 744 | 736 | 744 | 743 | 735 | 1 | 99.9%
08 | 744 | 736 | 744 | 680 | 672 | 64 | 91.3%
09 | 720 | 634 | 713 | 666 | 582 | 52 | 91.8%
10 | 744 | 735 | 743 | 743 | 734 | 1 | 99.9%
11 | 720 | 710 | 719 | 719 | 709 | 1 | 99.9%
12 | 744 | 731 | 744 | 744 | 731 | 0 | 100.0%

Total

### 4.3 Findings

There are 8,526 ozone hours and temperature in 8,404 of those hours (98.6%).
Most of the 122 hours without trial temperature are in August and September.
Ozone observations are present in every month.

Georgia has FM12 reports and quality code `7`, which were not in the Alabaster
file. FM12 is a report type; it enters this trial when its source/quality pair
qualifies. Quality code `7` is excluded by our trial rules.

In CP4, we will use these existing inputs for an early check of the shared
Spark preparation. The pairing is still provisional. The full Georgia rerun
comes later, after Alabama's Silver and Gold work.
